# Plateau’s Problem for Enneper surface

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from training.optimizers import GaussNewton 
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [2]:
# Parametric equations for Enneper's minimal surface in polar coordinates
def enneper_surface_polar(r, phi):
    x = r * torch.cos(phi) - (1/3) * r**3 * torch.cos(3 * phi)
    y = r * torch.sin(phi) + (1/3) * r**3 * torch.sin(3 * phi)
    z = r**2 * torch.cos(2 * phi)
    return x, y, z
    
def enneper_level_set(v):
    x = v[:, 0]
    y = v[:, 1]
    z = v[:, 2]
    
    term1 = y**2 - x**2 + (4/3)*z + (4/9)*z**3
    term2 = y**2 - x**2 + (8/9)*z - z*(x**2 + y**2 + (8/9)*z**2)
    output = term1**3 - 3*z*term2**2
    return output

def sample_true_surface(n_samples):
    # Generate radial and angular coordinates
    r = torch.linspace(-r_max, r_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, 2*n_samples, dtype=torch.float64)
    
    # Create a grid of r and phi
    r, phi = torch.meshgrid(r, phi, indexing='ij')

    # Compute the x, y, z coordinates using the parametric equations
    x, y, z = enneper_surface_polar(r.flatten(), phi.flatten())
    points_on_surface = torch.vstack([x, y, z]).T

    return points_on_surface

# Bounds and number of samples
n = 1000
r_max = 0.9
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)

# Generate boundary points with r constant and phi ranging from -pi to pi
r_constant = torch.full_like(phi, r_max, dtype=torch.float64)
x, y, z = enneper_surface_polar(r_constant, phi)
pts_boundary = torch.vstack([x, y, z]).T
pts_surface_true = sample_true_surface(64)

# Generate the mesh using the level set function
verts, faces = get_mesh(
    enneper_level_set, N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -2], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 2], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()


/cluster/home/jaking/.local/lib/python3.11/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

### Pretraining

In [3]:
# Generate random training points
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


# Define pretraining loss
def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    targets = pts[:, 2].to(dtype=torch.float64)
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss = {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-1.5, -1.5, -1], dtype=torch.float64),
    bbox_max=torch.tensor([1.5, 1.5, 1], dtype=torch.float64),
    chunks=2
)


fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()


Pretrain Iter 0: Loss = 0.293926
Pretrain Iter 100: Loss = 0.000395
Pretrain Iter 200: Loss = 0.000327
Pretrain Iter 300: Loss = 0.000275
Pretrain Iter 400: Loss = 0.000229
Pretrain Iter 500: Loss = 0.000191
Pretrain Iter 600: Loss = 0.000160
Pretrain Iter 700: Loss = 0.000136
Pretrain Iter 800: Loss = 0.000117
Pretrain Iter 900: Loss = 0.000103
Pretraining completed!


Output()

### Main training loop

In [4]:
pts_space = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_eikonal, pts_boundary))
pts_surface = pts_boundary


# GN weights
loss_weights = {"interface": 1.0, "eikonal": 0.001, "curvature": 1.0}
# Adam weights
# loss_weights = {"interface": 1.0, "eikonal": 0.1, "curvature": 1.0}


config = {
    "pts_boundary": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

In [5]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')
# Gauss-Newton:
optim = GaussNewton(model, lr=1e-1, config=config)
# Adam:
# optim = torch.optim.Adam(model.parameters(), lr=1e-3)
# eikonal weight has to be small compared to h_laplace, see StabEik
start_time = time.time()
current_time = 0
for i in (pbar:=trange(100000)):
    if current_time > 1200:
        break
    optim.zero_grad()

    loss_interface  = 0.5*model.f(params, pts_boundary).square().mean()

    loss_eikonal  = 0.5*model.r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

    if i == 500:
        loss_weights = {"interface": 1.0, "eikonal": 0.0, "curvature": 1.0}
        config["loss_weights"] = loss_weights
        optim.config = config

    pts_surface = sample_model_surface_binsearch(model, pts_boundary)
    config["pts_surface"] = pts_surface
    optim.config = config
    
    loss_curvature = 0.5*model.r_mean_curvature(params, pts_surface).squeeze(1).square().mean()
       
    loss = loss_weights["interface"] * loss_interface + loss_weights["eikonal"] * loss_eikonal + loss_weights["curvature"] * loss_curvature

    loss.backward()
    with torch.no_grad():
        loss_metric = loss_interface + loss_curvature
        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), enneper_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        pbar.set_description(f"interface: {loss_interface.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"curvature: {loss_curvature.item():.2e} "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"{len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

  0%|          | 0/100000 [00:00<?, ?it/s]

/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/torch/functional.py:512: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3587.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


KeyboardInterrupt: 

### Visualize the result

In [30]:
# Generate the mesh using the level set function
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-0.85, -0.85, -0.85], dtype=torch.float64),
    bbox_max=torch.tensor([0.85, 0.85, 0.85], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()

Output()

In [31]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-0.85, -0.85, -0.85], dtype=torch.float64),
    bbox_max=torch.tensor([0.85, 0.85, 0.85], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = model.r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

Output()